# Hybrid Recommendation System
## Combining Content-Based + Collaborative Filtering

This notebook combines two recommendation approaches:
1. **Content-Based**: Uses genre/title features (works for 100% of users)
2. **Collaborative Filtering**: Uses reader overlap (works for 22% of users)

### Hybrid Strategy
- **Score Calculation**: 50% Content-Based + 50% Collaborative Filtering
- **Benefit**: Diverse recommendations leveraging both signals
- **Fallback**: When CF has no signal (0 shared readers), content-based dominates

### System Architecture
1. Load content-based recommendation system
2. Load collaborative filtering system
3. Get recommendations from both
4. Merge by blending scores
5. Return combined top N recommendations

## Section 1: Load and Setup Both Recommendation Systems

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("Loading hybrid recommendation system...")
print("This requires both content-based and CF systems to be initialized.")
print()
print("Step 1: Loading data...")

# Load the user-book data
user_books_df = pd.read_csv('company_u.csv', header=0)
user_books_df.columns = ['User', 'Books']

# Parse books
def parse_books(books_string):
  if pd.isna(books_string):
    return []
  books_str = str(books_string)
  books = books_str.split(' / ')
  titles = []
  for book in books:
    if ' by ' in book:
      title = book.split(' by ')[0].strip()
    elif ' ; ' in book:
      title = book.split(' ; ')[0].strip()
    else:
      title = book.strip()
    title = title.rstrip('.')
    if title:
      titles.append(title)
  return titles

user_book_pairs = []
for _, row in user_books_df.iterrows():
  user = row['User']
  books = parse_books(row['Books'])
  for book in books:
    user_book_pairs.append({'User': user, 'Title': book})

user_books_parsed = pd.DataFrame(user_book_pairs)
unique_books = sorted(user_books_parsed['Title'].unique())
unique_users = sorted(user_books_parsed['User'].unique())

book_to_idx = {book: idx for idx, book in enumerate(unique_books)}
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}

rows = [user_to_idx[row['User']] for _, row in user_books_parsed.iterrows()]
cols = [book_to_idx[row['Title']] for _, row in user_books_parsed.iterrows()]
data = [1] * len(user_books_parsed)

user_book_matrix = csr_matrix((data, (rows, cols)), shape=(len(unique_users), len(unique_books)))

print(f" Users: {len(unique_users)}")
print(f" Books: {len(unique_books)}")
print(f" Interactions: {len(user_books_parsed)}")
print()
print("Data loaded successfully")

Loading hybrid recommendation system...
This requires both content-based and CF systems to be initialized.

Step 1: Loading data...
 Users: 298
 Books: 909
 Interactions: 999

Data loaded successfully


## Section 2: Initialize Collaborative Filtering System

In [2]:
print("Step 2: Setting up Collaborative Filtering...")

# Build co-occurrence matrix
co_occurrence_matrix = user_book_matrix.T @ user_book_matrix
co_occurrence_dense = co_occurrence_matrix.toarray()

# Jaccard similarity
def jaccard_similarity(co_occur_matrix):
  book_counts = np.array(co_occur_matrix.diagonal())
  jaccard = co_occur_matrix / (book_counts[:, None] + book_counts[None, :] - co_occur_matrix + 1e-8)
  return jaccard

jaccard_sim = jaccard_similarity(co_occurrence_dense)
cosine_sim = cosine_similarity(user_book_matrix.T)
cf_similarity_matrix = 0.6 * jaccard_sim + 0.4 * cosine_sim
np.fill_diagonal(cf_similarity_matrix, 0)

print("CF similarity matrix ready")
print()

def get_cf_recommendations_internal(user_id, num_recommendations=10):
  """Internal CF recommendation function"""
  if user_id not in user_to_idx:
    return {}
  
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  
  if len(read_book_indices) == 0:
    return {}
  
  unread_indices = np.where(user_vector == 0)[0]
  book_scores = {}
  
  for unread_idx in unread_indices:
    similarities = cf_similarity_matrix[unread_idx][read_book_indices]
    if len(similarities) > 0:
      avg_sim = np.mean(similarities)
      max_sim = np.max(similarities)
      score = 0.6 * max_sim + 0.4 * avg_sim
      book_scores[unread_idx] = score
  
  sorted_books = sorted(book_scores.items(), key=lambda x: x[1], reverse=True)
  return {unique_books[idx]: score for idx, score in sorted_books[:num_recommendations]}

print("CF recommendation function ready")

Step 2: Setting up Collaborative Filtering...
CF similarity matrix ready

CF recommendation function ready


## Section 3: Load Content-Based System from External Notebook

In [3]:
print("Step 3: Loading Content-Based recommendation system...")
print()

try:
  # Try to load from the content-based notebook state
  # This assumes the content-based notebook has been run and its variables are available
  print("NOTE: Content-based system should be loaded from company_u_content_based_recsys.ipynb")
  print()
  print("For now, we'll create a simplified content-based system for demonstration.")
  print("For production hybrid use, open BOTH notebooks and cross-reference.")
  print()
  
  # We'll create a simple TF-IDF based content system
  from sklearn.feature_extraction.text import TfidfVectorizer
  
  # For now, create content-based similarity using book titles only
  # In production, this would use full genre metadata
  vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,3), max_features=500)
  tfidf_matrix = vectorizer.fit_transform(unique_books)
  cb_similarity_matrix = cosine_similarity(tfidf_matrix)
  np.fill_diagonal(cb_similarity_matrix, 0)
  
  print(f"Content-Based similarity matrix shape: {cb_similarity_matrix.shape}")
  print(f"Mean similarity: {cb_similarity_matrix[cb_similarity_matrix > 0].mean():.4f}")
  print()
  print("Content-Based system ready (title-based TF-IDF)")
  print()
  print("NOTE: For better results, use the full content-based system from")
  print("   company_u_content_based_recsys.ipynb with genre features.")
  
except Exception as e:
  print(f"Warning: Could not load full content-based system: {e}")
  print("Using simplified title-based system instead.")

Step 3: Loading Content-Based recommendation system...

NOTE: Content-based system should be loaded from company_u_content_based_recsys.ipynb

For now, we'll create a simplified content-based system for demonstration.
For production hybrid use, open BOTH notebooks and cross-reference.

Content-Based similarity matrix shape: (909, 909)
Mean similarity: 0.1343

Content-Based system ready (title-based TF-IDF)

NOTE: For better results, use the full content-based system from
   company_u_content_based_recsys.ipynb with genre features.


In [4]:
def get_cb_recommendations_internal(user_id, num_recommendations=10):
  """Internal content-based recommendation function"""
  if user_id not in user_to_idx:
    return {}
  
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  
  if len(read_book_indices) == 0:
    return {}
  
  unread_indices = np.where(user_vector == 0)[0]
  book_scores = {}
  
  # For each unread book, compute similarity to read books
  for unread_idx in unread_indices:
    similarities = cb_similarity_matrix[unread_idx][read_book_indices]
    if len(similarities) > 0:
      avg_sim = np.mean(similarities)
      max_sim = np.max(similarities)
      score = 0.6 * max_sim + 0.4 * avg_sim
      book_scores[unread_idx] = score
  
  sorted_books = sorted(book_scores.items(), key=lambda x: x[1], reverse=True)
  return {unique_books[idx]: score for idx, score in sorted_books[:num_recommendations]}

print("Content-based recommendation function ready")

Content-based recommendation function ready


## Section 4: Implement Hybrid Recommendation Function

In [5]:
def get_hybrid_recommendations(user_id, num_recommendations=10, cb_weight=0.5, cf_weight=0.5):
  """
  Get hybrid recommendations combining Content-Based and Collaborative Filtering.
  
  Algorithm:
  1. Get recommendations from content-based system
  2. Get recommendations from collaborative filtering system
  3. Combine scores: hybrid_score = cb_weight * cb_score + cf_weight * cf_score
  4. Return top N merged recommendations
  
  Args:
    user_id (str): User ID (e.g., 'User162')
    num_recommendations (int): Number to return (default: 10)
    cb_weight (float): Weight for content-based (default: 0.5)
    cf_weight (float): Weight for collaborative filtering (default: 0.5)
  
  Returns:
    dict: Contains user info and hybrid recommendations dataframe
  """
  
  if user_id not in user_to_idx:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'{user_id} not found in dataset'
    }
  
  # Normalize weights
  total_weight = cb_weight + cf_weight
  cb_weight = cb_weight / total_weight
  cf_weight = cf_weight / total_weight
  
  # Get recommendations from both systems
  cb_recs = get_cb_recommendations_internal(user_id, num_recommendations=num_recommendations*2)
  cf_recs = get_cf_recommendations_internal(user_id, num_recommendations=num_recommendations*2)
  
  if not cb_recs and not cf_recs:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'No recommendations available for {user_id}'
    }
  
  # Combine scores
  all_books = set(cb_recs.keys()) | set(cf_recs.keys())
  hybrid_scores = {}
  
  for book in all_books:
    cb_score = cb_recs.get(book, 0)
    cf_score = cf_recs.get(book, 0)
    hybrid_score = cb_weight * cb_score + cf_weight * cf_score
    hybrid_scores[book] = {
      'score': hybrid_score,
      'cb_score': cb_score,
      'cf_score': cf_score
    }
  
  # Sort by hybrid score
  sorted_recs = sorted(hybrid_scores.items(), key=lambda x: x[1]['score'], reverse=True)
  top_recs = sorted_recs[:num_recommendations]
  
  # Create recommendations dataframe
  recs_data = []
  for rank, (book, scores) in enumerate(top_recs, 1):
    recs_data.append({
      'Rank': rank,
      'Title': book,
      'Hybrid Score': scores['score'],
      'Content-Based': scores['cb_score'],
      'Collaborative': scores['cf_score']
    })
  
  recs_df = pd.DataFrame(recs_data)
  
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  read_books = [unique_books[idx] for idx in read_book_indices]
  
  return {
    'user_id': user_id,
    'status': 'success',
    'num_books_read': len(read_books),
    'books_read': read_books[:5],
    'recommendations': recs_df,
    'cb_weight': cb_weight,
    'cf_weight': cf_weight
  }

print("Hybrid recommendation function ready")

Hybrid recommendation function ready


## Section 5: Test Hybrid System

In [6]:
# Test the hybrid system
test_user = 'User002'
result = get_hybrid_recommendations(test_user, num_recommendations=10)

if result['status'] == 'success':
  print(f"Hybrid Recommendations for {result['user_id']}")
  print(f"Books read: {result['num_books_read']}")
  print(f"\nWeighting: {result['cb_weight']*100:.0f}% Content-Based + {result['cf_weight']*100:.0f}% Collaborative")
  print(f"\nTop {len(result['recommendations'])} Recommendations:")
  print(result['recommendations'].to_string(index=False))
else:
  print(f"Error: {result['message']}")

Hybrid Recommendations for User002
Books read: 2

Weighting: 50% Content-Based + 50% Collaborative

Top 10 Recommendations:
 Rank                                                                                                                  Title  Hybrid Score  Content-Based  Collaborative
    1                                       An introduction to project management : predictive, agile, and hybrid approaches      0.261621       0.523242              0
    2                                                           John C. Mowen., Exploring Web marketing & project management      0.244588       0.489176              0
    3                                                                             Financial management : theory and practice      0.230865       0.461731              0
    4                                                                           Marketing management and strategy : a reader      0.210111       0.420223              0
    5                          

## Section 6: Interactive Testing Interface

In [7]:
all_users = list(unique_users)

print(f"Total users in dataset: {len(all_users)}")
print(f"User ID range: {all_users[0]} to {all_users[-1]}")

def display_hybrid_recommendations(user_id, num_recs=10, cb_weight=0.5, cf_weight=0.5):
  """Display hybrid recommendations nicely"""
  result = get_hybrid_recommendations(user_id, num_recommendations=num_recs, 
                    cb_weight=cb_weight, cf_weight=cf_weight)
  
  if result['status'] == 'error':
    print(f"Error: {result['message']}")
    return
  
  print(f"\n{'='*100}")
  print(f"HYBRID RECOMMENDATIONS FOR {result['user_id']}")
  print(f"{'='*100}")
  print(f"Books user has read: {result['num_books_read']}")
  print(f"\nWeighting: {result['cb_weight']*100:.0f}% Content-Based + {result['cf_weight']*100:.0f}% Collaborative")
  print(f"\nSample books already read:")
  for book in result['books_read']:
    print(f" {book}")
  
  print(f"\n{'-'*100}")
  print(f"TOP {len(result['recommendations'])} HYBRID RECOMMENDATIONS:")
  print(f"{'-'*100}")
  
  for _, row in result['recommendations'].iterrows():
    print(f"#{int(row['Rank']):2d}. {row['Title'][:60]}")
    print(f"   Hybrid: {row['Hybrid Score']:.4f} = {row['Content-Based']:.4f} (CB) + {row['Collaborative']:.4f} (CF)")
    print()

print("Display function created")

Total users in dataset: 298
User ID range: User002 to User300
Display function created


In [8]:
# Interactive testing with widgets
try:
  from ipywidgets import Dropdown, IntSlider, FloatSlider, Button, Output, VBox, HBox
  from IPython.display import display, clear_output
  
  # Create widgets
  user_dropdown = Dropdown(
    options=all_users,
    value=all_users[0],
    description='Select User:',
    style={'description_width': '120px'}
  )
  
  num_recs_slider = IntSlider(
    value=10,
    min=1,
    max=30,
    step=1,
    description='# Recommendations:',
    style={'description_width': '160px'}
  )
  
  cb_weight_slider = FloatSlider(
    value=0.5,
    min=0,
    max=1,
    step=0.1,
    description='Content-Based Weight:',
    style={'description_width': '160px'}
  )
  
  cf_weight_slider = FloatSlider(
    value=0.5,
    min=0,
    max=1,
    step=0.1,
    description='CF Weight:',
    style={'description_width': '160px'}
  )
  
  execute_button = Button(
    description='Show Recommendations',
    button_style='info',
    tooltip='Click to get hybrid recommendations'
  )
  
  output = Output()
  
  # Button callback
  def on_button_click(b):
    with output:
      clear_output(wait=True)
      selected_user = user_dropdown.value
      num_recs = num_recs_slider.value
      cb_wt = cb_weight_slider.value
      cf_wt = cf_weight_slider.value
      display_hybrid_recommendations(selected_user, num_recs=num_recs, 
                     cb_weight=cb_wt, cf_weight=cf_wt)
  
  execute_button.on_click(on_button_click)
  
  # Layout
  controls = VBox([
    user_dropdown,
    num_recs_slider,
    cb_weight_slider,
    cf_weight_slider,
    execute_button
  ])
  
  display(controls)
  display(output)
  
  # Auto-run on first load
  on_button_click(None)
  
except ImportError:
  print("\nipywidgets not available. Test using direct function calls above.\n")

Output()

## Section 7: Compare All Three Approaches

In [9]:
print("="*100)
print("COMPARING ALL THREE RECOMMENDATION APPROACHES")
print("="*100)

test_users = ['User002', 'User010', 'User050', 'User100']

for user_id in test_users:
  print(f"\n\n{'='*100}")
  print(f"USER: {user_id}")
  print(f"{'='*100}")
  
  # Content-Based
  cb_recs = get_cb_recommendations_internal(user_id, num_recommendations=5)
  
  # Collaborative Filtering
  cf_recs = get_cf_recommendations_internal(user_id, num_recommendations=5)
  
  # Hybrid
  hybrid_result = get_hybrid_recommendations(user_id, num_recommendations=5, cb_weight=0.5, cf_weight=0.5)
  
  print(f"\nCONTENT-BASED (Score: Content Similarity)")
  for i, (book, score) in enumerate(list(cb_recs.items())[:5], 1):
    print(f" {i}. {book[:55]:55} | Score: {score:.4f}")
  
  print(f"\nCOLLABORATIVE FILTERING (Score: Reader Overlap)")
  for i, (book, score) in enumerate(list(cf_recs.items())[:5], 1):
    print(f" {i}. {book[:55]:55} | Score: {score:.4f}")
  
  print(f"\nHYBRID (50% CB + 50% CF)")
  if hybrid_result['status'] == 'success':
    for _, row in hybrid_result['recommendations'].iterrows():
      print(f" {int(row['Rank'])}. {row['Title'][:55]:55} | Score: {row['Hybrid Score']:.4f}")
  
  print()

COMPARING ALL THREE RECOMMENDATION APPROACHES


USER: User002

CONTENT-BASED (Score: Content Similarity)
 1. An introduction to project management : predictive, agi | Score: 0.5232
 2. John C. Mowen., Exploring Web marketing & project manag | Score: 0.4892
 3. Financial management : theory and practice              | Score: 0.4617
 4. Marketing management and strategy : a reader            | Score: 0.4202
 5. Marketing management : analysis, planning, implementati | Score: 0.4200

COLLABORATIVE FILTERING (Score: Reader Overlap)
 1. $Pread : the best of the magazine that illuminated the  | Score: 0.0000
 2. 12 rules for life : an antidote to chaos                | Score: 0.0000
 3. 21st century reading. 2, Student book : creative thinki | Score: 0.0000
 4. A collection of essays                                  | Score: 0.0000
 5. A concise history of the Armenian people : (from ancien | Score: 0.0000

HYBRID (50% CB + 50% CF)
 1. An introduction to project management : predictive, agi 

## Section 8: Summary and Analysis

In [10]:
print("="*100)
print("HYBRID SYSTEM SUMMARY")
print("="*100)

print(f"""
SYSTEM READY

Three Recommendation Approaches Available:

1. CONTENT-BASED (company_u_content_based_recsys.ipynb)
  • Uses: Genre/Title features
  • Coverage: 100% of users
  • Pros: Reliable, consistent, works for everyone
  • Cons: Only uses item features, ignores user patterns

2. COLLABORATIVE FILTERING (company_u_collaborative_filtering.ipynb)
  • Uses: Reader overlap / co-occurrence
  • Coverage: 22% of users (isolated tastes not captured)
  • Pros: Captures implicit user preferences
  • Cons: Data too sparse, fails on unique tastes

3. HYBRID (this notebook)
  • Uses: 50% Content-Based + 50% Collaborative
  • Coverage: 100% of users (CB fallback covers everyone)
  • Pros: Best of both worlds
  • Cons: Slightly more complex

 RECOMMENDED WEIGHTS:
  • Equal blending (50/50): Balanced signal from both
  • Content-heavy (70/30): Trust genre more, use CF as signal boost
  • CF-heavy (30/70): Emphasize reader patterns, fall back to content

 NEXT STEPS:
  1. Test hybrid with different weight configurations
  2. A/B test recommendations with actual users
  3. When adding neural network taste model:
    → Create ensemble with 33% each approach
    → Or: (Content + CF + Neural) / 3

🔧 CUSTOMIZATION:
  Change weights in any function call:
    get_hybrid_recommendations(user_id, num_recommendations=10,
                 cb_weight=0.7, cf_weight=0.3)
""")

HYBRID SYSTEM SUMMARY

SYSTEM READY

Three Recommendation Approaches Available:

1. CONTENT-BASED (company_u_content_based_recsys.ipynb)
  • Uses: Genre/Title features
  • Coverage: 100% of users
  • Pros: Reliable, consistent, works for everyone
  • Cons: Only uses item features, ignores user patterns

2. COLLABORATIVE FILTERING (company_u_collaborative_filtering.ipynb)
  • Uses: Reader overlap / co-occurrence
  • Coverage: 22% of users (isolated tastes not captured)
  • Pros: Captures implicit user preferences
  • Cons: Data too sparse, fails on unique tastes

3. HYBRID (this notebook)
  • Uses: 50% Content-Based + 50% Collaborative
  • Coverage: 100% of users (CB fallback covers everyone)
  • Pros: Best of both worlds
  • Cons: Slightly more complex

 RECOMMENDED WEIGHTS:
  • Equal blending (50/50): Balanced signal from both
  • Content-heavy (70/30): Trust genre more, use CF as signal boost
  • CF-heavy (30/70): Emphasize reader patterns, fall back to content

 NEXT STEPS:
  1. Tes